In [30]:
import sympy as sp

In [31]:
from functools import lru_cache

@lru_cache(maxsize=None)
def get_ckn(k: int, n: int, p):
    if k<0 or k>n:
        return sp.Integer(0)
    if n==0:
        return sp.Integer(1) if k == 0 else sp.Integer(0)
    return get_ckn(k-1, n-1, p)/(2*p) + (k+1)*get_ckn(k+1, n-1, p)

In [32]:
#übungsaufe Fiboniacci rehe F(40) rekusiv implemtieren, einmal mit memoritzation ein ohne

In [33]:
x = sp.symbols('x', real = True)
alpha = sp.symbols('alpha', real = True, positive = True)
Ax = sp.symbols('Ax', real = True)

In [34]:
g0 = sp.exp(-alpha*(x-Ax)**2  )
g0

exp(-alpha*(-Ax + x)**2)

In [35]:
def get_hermite_gaussian(k):
    return g0.diff(Ax, k)


In [36]:
get_hermite_gaussian(20).simplify()

alpha**10*(1048576*alpha**10*(Ax - x)**20 - 99614720*alpha**9*(Ax - x)**18 + 3810263040*alpha**8*(Ax - x)**16 - 76205260800*alpha**7*(Ax - x)**14 + 866834841600*alpha**6*(Ax - x)**12 - 5721109954560*alpha**5*(Ax - x)**10 + 21454162329600*alpha**4*(Ax - x)**8 - 42908324659200*alpha**3*(Ax - x)**6 + 40226554368000*alpha**2*(Ax - x)**4 - 13408851456000*alpha*(Ax - x)**2 + 670442572800)*exp(-alpha*(Ax - x)**2)

In [37]:
nmax = 6
for n in range(nmax + 1):
    g = 0
    for k in range(0, n+1):
        g = g + get_ckn(k, n, alpha)*get_hermite_gaussian(k)
    g = g.simplify()
    print(f"g({n}) = {g}")

g(0) = exp(-alpha*(Ax - x)**2)
g(1) = (-Ax + x)*exp(-alpha*(Ax - x)**2)
g(2) = (Ax - x)**2*exp(-alpha*(Ax - x)**2)
g(3) = -Ax*(Ax - x)**2*exp(-alpha*(Ax - x)**2) + x*(Ax - x)**2*exp(-alpha*(Ax - x)**2)
g(4) = (Ax - x)**4*exp(-alpha*(Ax - x)**2)
g(5) = -Ax*(Ax - x)**4*exp(-alpha*(Ax - x)**2) + x*(Ax - x)**4*exp(-alpha*(Ax - x)**2)
g(6) = (Ax - x)**6*exp(-alpha*(Ax - x)**2)


## Implementierung der Überlappintegrale

In [40]:
alpha, beta = sp.symbols('alpha beta', real = True, positive = True)
Ax, Ay, Az = sp.symbols('Ax Ay Az', real = True)
Bx, By, Bz = sp.symbols('Bx By Bz', real = True)

p = alpha + beta
q = alpha*beta/p
RAB2 = (Ax-Bx)**2 + (Ay-By)**2 + (Az-Bz)**2

In [41]:
S00 = (sp.sqrt(sp.pi/p))**3 * sp.exp(-q*RAB2)
S00

pi**(3/2)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(3/2)

In [62]:
def build_derivative_table(base_integral, Lmax, Avars, Bvars):
    Ax, Ay, Az = Avars
    Bx, By, Bz = Bvars

    all_idx = [(i, j, L- i -j) for L in range(Lmax+1)
                                for i in range (L +1) 
                                for j in range(L + 1- i)]

    @lru_cache(maxsize=None)
    def derivative(i,j,k,l,m,n):
        if (i,j,k,l,m,n) == (0,0,0,0,0,0):
            return base_integral
        if i > 0:
            return sp.diff(derivative(i-1, j, k, l, m, n),Ax)
        if j > 0:
            return sp.diff(derivative(i, j-1, k, l, m, n),Ay)
        if k > 0:
            return sp.diff(derivative(i, j, k-1, l, m, n),Az)
        if l > 0:
            return sp.diff(derivative(i, j, k, l-1, m, n),Bx)
        if m > 0:
            return sp.diff(derivative(i, j, k, l, m-1, n),By)
        return sp.diff(derivative(i, j, k, l, m, n-1),Bz)

    derivatives_dict = {}
    for (i,j,k) in all_idx:
        for (l, m, n) in all_idx:
            derivatives_dict[(i,j,k,l,m,n)] = sp.simplify(derivative(i,j,k,l,m,n))
    return derivatives_dict

In [63]:
derivative(1,0,0,0,0,0)

-pi**(3/2)*alpha*beta*(2*Ax - 2*Bx)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(5/2)

In [68]:
Lmax = 2

derivatives_dict = build_derivative_table(S00, Lmax, (Ax, Ay, Az), (Bx, By, Bz))


In [70]:
for key, value in derivatives_dict.items():
    print(key, value)

(0, 0, 0, 0, 0, 0) pi**(3/2)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(3/2)
(0, 0, 0, 0, 0, 1) 2*pi**(3/2)*alpha*beta*(Az - Bz)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(5/2)
(0, 0, 0, 0, 1, 0) 2*pi**(3/2)*alpha*beta*(Ay - By)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(5/2)
(0, 0, 0, 1, 0, 0) 2*pi**(3/2)*alpha*beta*(Ax - Bx)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(5/2)
(0, 0, 0, 0, 0, 2) 2*pi**(3/2)*alpha*beta*(2*alpha*beta*(Az - Bz)**2 - alpha - beta)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(7/2)
(0, 0, 0, 0, 1, 1) 4*pi**(3/2)*alpha**2*beta**2*(Ay - By)*(Az - Bz)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(7/2)
(0, 0, 0, 0, 2, 0) 2*pi**(3/2)*alpha*beta*(2*alpha*beta*(Ay - By)**2